Dans ce notebook, nous avons tenté d'implémenter différentes méthodes de ML, dont un random Forest notamment.

Notre but est de prédire la valorisation d'une start-up après la n+1-ème levée de fond, à partir de ses valorisations précédentes, et d'autres paramètres tels que la date des différents rounds.

Nous avons utilisé un jeu de données simulé avec Claude, car nous ne disposions pas d'un jeu de données réel avec suffisamment d'entreprises.

Les données réelles de levées de fonds, les tables de capitalisation et les valorisations post-money sont hautement confidentielles et rarement structurées dans des bases de données publiques massives. Pour contourner cette limitation tout en conservant une approche rigoureuse, nous avons généré un jeu de données synthétique.

Cette approche permet de contrôler la distribution probabiliste des variables et de simuler des dynamiques de marché réalistes. L'analyse exploratoire initiale valide la cohérence de notre dataset : la croissance de la valorisation entre deux tours s'établit à 2.43x, et la proportion des baisses de valorisation est de 8.2%. Ces métriques sont conformes aux observations macroéconomiques actuelles du marché de la tech, ce qui garantit que notre modèle s'entraînera sur un environnement financier plausible.

Nous avons d'abord importé toutes les bibliothèques nécessaires pour faire du machine learning, et pour générer notre jeu de données

In [ ]:
import platform

print(f"Python version: {platform.python_version()}")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print(f"NumPy version: {np.__version__}")

Python version: 3.12.13
NumPy version: 2.0.2


In [ ]:
# Setup plots
%matplotlib inline
plt.rcParams["figure.figsize"] = 10, 8
%config InlineBackend.figure_format = "retina"
sns.set()

In [ ]:
import sklearn

print(f"scikit-learn version: {sklearn.__version__}")

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

import joblib  # For saving models to disk

scikit-learn version: 1.6.1


Voici ensuite la génération du jeu de données (entièrement faite avec Claude, comme nous l'a suggéré notre encadrant)

In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
Générateur de dataset synthétique de levées de fonds (Venture Capital)
pour la prédiction de la valorisation Post-Money au tour `n`.
================================================================================
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd


# ------------------------------------------------------------------------------
# 1. CONFIGURATION (tout est ajustable ici)
# ------------------------------------------------------------------------------
@dataclass
class Config:
    # --- Volume cible ---
    n_rows: int = 10_000  # nb de lignes d'entraînement ciblées APRÈS filtrage

    # --- Nombre de tours par société (entre 4 et 15) ---
    min_rounds: int = 4
    max_rounds: int = 15
    rounds_decay: float = 0.30  # + élevé => sociétés concentrées sur peu de tours

    # --- Valorisation de départ (tour 1, type Pre-Seed) en M$ ---
    seed_val_median: float = 5.0  # médiane lognormale
    seed_val_sigma: float = 0.45  # dispersion lognormale
    seed_val_clip: tuple = (2.0, 12.0)

    # --- Facteur de croissance "up round" (step-up) ---
    growth_min: float = 1.5  # borne basse demandée
    growth_max: float = 3.0  # borne haute demandée
    growth_mean_early: float = 2.9  # moyenne step-up aux premiers tours (Carta ~2.6-2.8x)
    growth_mean_late: float = 1.9  # moyenne step-up aux tours avancés (compression late stage)
    growth_spread: float = 0.35  # écart-type autour de la moyenne par stade

    # --- Down rounds (facteur de croissance < 1) ---
    downround_prob: float = 0.08  # 5-10% demandé (réalité 2024-25 ~0.19-0.23)
    downround_min: float = 0.50  # une down round perd au plus 50% de la valo
    downround_max: float = 0.95  # ... au moins 5%

    # --- Dilution (Amount_Raised_M / Post_Val_M) ---
    dilution_min: float = 0.10  # 10% demandé
    dilution_max: float = 0.25  # 25% demandé

    # --- Sortie ---
    output_csv: str = "vc_synthetic_funding_dataset.csv"
    random_seed: int = 42  # reproductibilité


# Échelle ORDONNÉE des stades : round_number (1-indexé) -> Deal_Type.
STAGE_LADDER = [
    "Pre-Seed", "Seed",
    "Series A", "Series B", "Series C", "Series D", "Series E",
    "Series F", "Series G", "Series H", "Series I", "Series J",
    "Series K", "Series L", "Series M",
]


# ------------------------------------------------------------------------------
# 2. BRIQUES DE GÉNÉRATION
# ------------------------------------------------------------------------------
def _stage_growth_mean(round_number: int, cfg: Config) -> float:
    """Moyenne du step-up attendu pour un tour donné."""
    first_growth_round, last_growth_round = 2, 8
    span = last_growth_round - first_growth_round
    frac = np.clip((round_number - first_growth_round) / span, 0.0, 1.0)
    mean = cfg.growth_mean_early - (cfg.growth_mean_early - cfg.growth_mean_late) * frac
    return float(np.clip(mean, cfg.growth_min, cfg.growth_max))


def _sample_growth_factor(round_number: int, rng: np.random.Generator, cfg: Config):
    """Tire un facteur de croissance pour passer du tour n-1 au tour n."""
    if rng.random() < cfg.downround_prob:
        return float(rng.uniform(cfg.downround_min, cfg.downround_max)), True

    mean = _stage_growth_mean(round_number, cfg)
    factor = rng.normal(mean, cfg.growth_spread)
    return float(np.clip(factor, cfg.growth_min, cfg.growth_max)), False


def _sample_n_rounds(rng: np.random.Generator, cfg: Config) -> int:
    """Nombre de tours pour une société."""
    choices = np.arange(cfg.min_rounds, cfg.max_rounds + 1)
    weights = np.exp(-cfg.rounds_decay * (choices - cfg.min_rounds))
    weights /= weights.sum()
    return int(rng.choice(choices, p=weights))


def _get_deal_type(round_index: int) -> str:
    """Récupère le nom du stade (Deal_Type) de manière robuste."""
    if round_index < len(STAGE_LADDER):
        return STAGE_LADDER[round_index]
    # Fallback propre pour générer Series N, O, P... si max_rounds > 15
    letter = chr(ord('A') + round_index - 2)
    return f"Series {letter}"


def _generate_company(company_idx: int, rng: np.random.Generator, cfg: Config):
    """Génère la série temporelle complète d'UNE société."""
    n_rounds = _sample_n_rounds(rng, cfg)

    # --- Tour 1 : valorisation initiale ---
    start_val = float(np.clip(
        rng.lognormal(mean=np.log(cfg.seed_val_median), sigma=cfg.seed_val_sigma),
        *cfg.seed_val_clip,
    ))
    post_vals = [round(start_val, 2)]
    growth_factors = [np.nan]  # pas de step-up pour le tour 1
    is_down = [False]

    # --- Tours suivants : compounding via le facteur de croissance ---
    for i in range(1, n_rounds):
        round_number = i + 1
        factor, down = _sample_growth_factor(round_number, rng, cfg)
        post_vals.append(round(post_vals[-1] * factor, 2))
        growth_factors.append(round(factor, 4))
        is_down.append(down)

    # --- Construction des lignes ---
    company_id = f"C{company_idx:05d}"
    cumulative_raised = 0.0
    rows = []

    for i in range(n_rounds):
        dilution = rng.uniform(cfg.dilution_min, cfg.dilution_max)
        amount_raised = round(post_vals[i] * dilution, 2)
        cumulative_raised = round(cumulative_raised + amount_raised, 2)

        rows.append({
            "Company_ID": company_id,
            "Round_Number": i + 1,
            "Deal_Type": _get_deal_type(i),
            "Amount_Raised_M": amount_raised,
            "Total_Raised_to_Date_M": cumulative_raised,
            "Prev_Post_Val_M": post_vals[i - 1] if i > 0 else np.nan,  # NaN au tour 1
            "Post_Val_M": post_vals[i],  # <-- TARGET
            "Round_Growth_Factor": growth_factors[i],
            "Is_Down_Round": is_down[i],
        })

    return rows, n_rounds - 1


# ------------------------------------------------------------------------------
# 3. ASSEMBLAGE DU DATASET
# ------------------------------------------------------------------------------
def generate_dataset(cfg: Config | None = None) -> pd.DataFrame:
    """
    Génère des sociétés jusqu'à atteindre `n_rows` lignes entrainables.
    Afin de ne pas briser la série temporelle d'une entreprise, on ne coupe
    pas la dernière entreprise générée (le dataset final dépassera donc
    légèrement n_rows).
    """
    cfg = cfg or Config()
    rng = np.random.default_rng(cfg.random_seed)

    all_rows: list[dict] = []
    trainable_rows = 0
    company_idx = 1

    while trainable_rows < cfg.n_rows:
        rows, n_trainable = _generate_company(company_idx, rng, cfg)
        all_rows.extend(rows)
        trainable_rows += n_trainable
        company_idx += 1

    df = pd.DataFrame(all_rows)

    # --- FILTRAGE : ne garder que les lignes prêtes pour X -> y ---
    df = df[df["Prev_Post_Val_M"].notna()].reset_index(drop=True)

    return df


# ------------------------------------------------------------------------------
# 4. POST-TRAITEMENT : DATES COHÉRENTES
# ------------------------------------------------------------------------------
def add_coherent_dates(df: pd.DataFrame, random_seed: int = 42) -> pd.DataFrame:
    """
    Ajoute des dates de levées de fonds chronologiques sans modifier la logique financière.
    """
    rng = np.random.default_rng(random_seed)

    prev_dates = []
    current_dates = []
    months_elapsed_list = []

    # On traite chaque entreprise séparément pour respecter sa propre continuité temporelle
    for company_id, group in df.groupby("Company_ID", sort=False):
        # Date du tout premier tour (Pre-Seed/Seed retiré du dataset) : entre 2014 et 2021
        start_year = rng.integers(2014, 2022)
        start_month = rng.integers(1, 13)
        start_day = rng.integers(1, 29) # Limité à 28 pour éviter les erreurs de fin de mois

        # Initialisation de la date du "tour précédent"
        current_prev_date = pd.Timestamp(year=start_year, month=start_month, day=start_day)

        for row in group.itertuples():
            round_num = row.Round_Number

            # Paramétrage réaliste du temps entre deux tours selon le stade (en mois)
            if round_num <= 2:
                months_between = rng.uniform(12, 18)  # Seed -> Series A
            elif round_num <= 4:
                months_between = rng.uniform(15, 24)  # Series A -> Series C
            else:
                months_between = rng.uniform(18, 36)  # Late stage (tours très espacés)

            days_between = int(months_between * 30.44)
            current_round_date = current_prev_date + pd.Timedelta(days=days_between)

            # Sauvegarde des dates pour cette ligne
            prev_dates.append(current_prev_date)
            current_dates.append(current_round_date)
            months_elapsed_list.append(round(months_between, 1))

            # La date actuelle devient la date précédente pour la prochaine boucle
            current_prev_date = current_round_date

    # Injection propre dans une copie du DataFrame
    df_out = df.copy()

    # Insertion des nouvelles colonnes à des emplacements pertinents
    df_out.insert(2, "Prev_Round_Date", prev_dates)
    df_out.insert(3, "Round_Date", current_dates)

    # BONUS ML : Le temps écoulé est une feature indispensable pour prédire une valorisation
    df_out.insert(4, "Months_Since_Last_Round", months_elapsed_list)

    # Formatage texte YYYY-MM-DD pour un export CSV propre
    df_out["Prev_Round_Date"] = df_out["Prev_Round_Date"].dt.strftime('%Y-%m-%d')
    df_out["Round_Date"] = df_out["Round_Date"].dt.strftime('%Y-%m-%d')

    return df_out


# ------------------------------------------------------------------------------
# 5. CONTRÔLE QUALITÉ / RÉSUMÉ
# ------------------------------------------------------------------------------
def print_summary(df: pd.DataFrame) -> None:
    """Affiche des statistiques de validation pour vérifier le réalisme du dataset."""
    implied_stepup = df["Post_Val_M"] / df["Prev_Post_Val_M"]
    implied_dilution = df["Amount_Raised_M"] / df["Post_Val_M"]

    print("=" * 70)
    print("RÉSUMÉ DU DATASET SYNTHÉTIQUE")
    print("=" * 70)
    print(f"Lignes (X -> y) : {len(df):,}")
    print(f"Sociétés uniques : {df['Company_ID'].nunique():,}")
    print(f"Tours / société (min-max): "
          f"{df.groupby('Company_ID')['Round_Number'].max().min()} - "
          f"{df.groupby('Company_ID')['Round_Number'].max().max()}")
    print(f"Part de down rounds : {df['Is_Down_Round'].mean():.1%}")
    print(f"Step-up médian (Post/Prev): {implied_stepup.median():.2f}x "
          f"(moyenne {implied_stepup.mean():.2f}x)")
    print(f"Dilution médiane : {implied_dilution.median():.1%} "
          f"(min {implied_dilution.min():.1%} / max {implied_dilution.max():.1%})")
    print(f"Valeurs manquantes Prev : {df['Prev_Post_Val_M'].isna().sum()} "
          f"(doit être 0)")

    print("\nPost_Val_M MÉDIAN par stade (vs benchmarks marché) :")
    bench = {
        "Seed": "12-20", "Series A": "40-79", "Series B": "120-160",
        "Series C": "300-500", "Series D": "460+",
    }
    med = df.groupby("Deal_Type")["Post_Val_M"].median()
    for stage in STAGE_LADDER:
        if stage in med.index:
            ref = f"(bench ~{bench[stage]} M$)" if stage in bench else ""
            print(f"  {stage:<10}: {med[stage]:>9.1f} M$ {ref}")

    print("\nAperçu (5 premières lignes) :")
    with pd.option_context("display.max_columns", None, "display.width", 120):
        print(df.head().to_string(index=False))
    print("=" * 70)


# ------------------------------------------------------------------------------
# 6. POINT D'ENTRÉE
# ------------------------------------------------------------------------------
def main() -> pd.DataFrame:
    cfg = Config()

    # 1. Génération des données financières
    df = generate_dataset(cfg)

    # 2. Ajout des dates chronologiques
    df = add_coherent_dates(df, random_seed=cfg.random_seed)

    # 3. Export CSV et affichage
    df.to_csv(cfg.output_csv, index=False)
    print_summary(df)
    print(f"\n[OK] Dataset enregistré : {cfg.output_csv}")

    return df


if __name__ == "__main__":
    main()

RÉSUMÉ DU DATASET SYNTHÉTIQUE
Lignes (X -> y) : 10,004
Sociétés uniques : 1,837
Tours / société (min-max): 4 - 15
Part de down rounds : 8.2%
Step-up médian (Post/Prev): 2.43x (moyenne 2.30x)
Dilution médiane : 17.6% (min 10.0% / max 25.1%)
Valeurs manquantes Prev : 0 (doit être 0)

Post_Val_M MÉDIAN par stade (vs benchmarks marché) :
  Seed      :      13.2 M$ (bench ~12-20 M$)
  Series A  :      33.0 M$ (bench ~40-79 M$)
  Series B  :      78.2 M$ (bench ~120-160 M$)
  Series C  :     174.3 M$ (bench ~300-500 M$)
  Series D  :     356.4 M$ (bench ~460+ M$)
  Series E  :     652.4 M$ 
  Series F  :    1187.4 M$ 
  Series G  :    1755.8 M$ 
  Series H  :    3317.6 M$ 
  Series I  :    6104.9 M$ 
  Series J  :   11411.8 M$ 
  Series K  :   17115.3 M$ 
  Series L  :   31911.7 M$ 
  Series M  :   39128.2 M$ 

Aperçu (5 premières lignes) :
Company_ID  Round_Number Prev_Round_Date Round_Date  Months_Since_Last_Round Deal_Type  Amount_Raised_M  Total_Raised_to_Date_M  Prev_Post_Val_M  Post_Va

Nous avons tout d'abord visualisé notre jeu de données, afin de voir quelles colonnes nous allons conserver ou non.

In [ ]:
df =pd.read_csv('vc_synthetic_funding_dataset.csv')
df.head(n=20)

,Company_ID,Round_Number,Prev_Round_Date,Round_Date,Months_Since_Last_Round,Deal_Type,Amount_Raised_M,Total_Raised_to_Date_M,Prev_Post_Val_M,Post_Val_M,Round_Growth_Factor,Is_Down_Round
0,C00001,2,2014-10-19,2016-03-24,17.2,Seed,1.03,1.60,3.13,9.39,3.0000,False
1,C00001,3,2016-03-24,2017-12-31,21.3,Series A,4.79,6.39,9.39,21.39,2.2776,False
2,C00001,4,2017-12-31,2019-04-27,15.8,Series B,10.23,16.62,21.39,52.53,2.4560,False
3,C00001,5,2019-04-27,2022-04-13,35.6,Series C,23.59,40.21,52.53,110.39,2.1014,False
4,C00001,6,2022-04-13,2024-12-02,31.7,Series D,42.37,82.58,110.39,276.59,2.5056,False
5,C00001,7,2024-12-02,2027-08-07,32.1,Series E,167.19,249.77,276.59,680.74,2.4612,False
6,C00001,8,2027-08-07,2029-04-16,20.3,Series F,254.71,504.48,680.74,1088.67,1.5992,False
7,C00002,2,2017-11-13,2019-01-20,14.2,Seed,2.88,3.59,4.27,12.81,3.0000,False
8,C00002,3,2019-01-20,2020-12-30,23.3,Series A,7.58,11.17,12.81,36.95,2.8841,False
9,C00002,4,2020-12-30,2022-09-23,20.8,Series B,13.16,24.33,36.95,89.58,2.4244,False


Avant toute manipulation, le jeu de données est scindé en un ensemble d'entraînement (80%) et un ensemble de test (20%). Cette ségrégation stricte est fondamentale pour prévenir la fuite de données.

Afin d'optimiser l'apprentissage, nous appliquons un pipeline de transformation adapté à la nature de chaque variable :

Variables numériques : Un StandardScaler est appliqué pour centrer et réduire les données. Bien que les arbres de décision soient théoriquement invariants à l'échelle, cette standardisation est cruciale pour les modèles linéaires (utilisés comme baseline) et stabilise numériquement le processus. Les valeurs manquantes éventuelles sont imputées par la médiane.

Variables catégorielles : Le stade de financement (Series A, B, etc.) est transformé via un OneHotEncoder.

In [ ]:
# Split dataset between training and test
df_train, df_test = train_test_split(df, test_size=0.2)

print(f"Training dataset: {df_train.shape}")
print(f"Test dataset: {df_test.shape}")

Training dataset: (8003, 12)
Test dataset: (2001, 12)


In [ ]:
# Split original training dataset between inputs and target
# (engineered features are not used)

# Target attribute is removed from training data
df_x_train = df_train.drop("Post_Val_M", axis=1)

# Targets are transformed into a NumPy array for further use during training
y_train = df_train["Post_Val_M"].to_numpy()

print(f"Training data: {df_x_train.shape}")
print(f"Training labels: {y_train.shape}")

Training data: (8003, 11)
Training labels: (8003,)


Nous excluons les colonnes qui ne nous servent pas

In [ ]:
# 1. On retire les colonnes identifiantes et temporelles brutes
colonnes_a_exclure = ["Company_ID", "Round_Date", "Prev_Post_Val_M"]

df_x_train_clean = df_x_train.drop(columns=colonnes_a_exclure)

# 2. On sépare intelligemment les colonnes restantes
# Variables numériques (montants, mois, facteurs)
num_features = df_x_train_clean.select_dtypes(include=[np.number]).columns
print(f"Features numériques : {list(num_features)}")

# Variables catégorielles (Deal_Type, Is_Down_Round)
cat_features = df_x_train_clean.select_dtypes(exclude=[np.number]).columns
print(f"Features catégorielles : {list(cat_features)}")

Features numériques : ['Round_Number', 'Months_Since_Last_Round', 'Amount_Raised_M', 'Total_Raised_to_Date_M', 'Round_Growth_Factor']
Features catégorielles : ['Prev_Round_Date', 'Deal_Type', 'Is_Down_Round']


Nous nettoyons le jeu de donnés à l'aide de Pipeline (remplacer les valeurs manquantes par la médiane, One Hot encoding, normalisation)

In [ ]:
# Pipeline pour les nombres (imputation + mise à l'échelle)
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("std_scaler", StandardScaler())
])

# Pipeline pour le texte/booléens (imputation + encodage)
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Assemblage final
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

# IMPORTANT : On applique le pipeline sur les données NETTOYÉES
x_train = full_pipeline.fit_transform(df_x_train_clean)

# Affichage des nouvelles dimensions
print(f"x_train dimensions: {x_train.shape}")
print("Aperçu de la première ligne :")
print(x_train[0].round(2))

x_train dimensions: (8003, 4628)
Aperçu de la première ligne :
[-1.11 -1.17 -0.16 ...  0.    1.    0.  ]


Pour évaluer la complexité de notre problème, nous avons implémenté deux modèles simples.

La Régression Linéaire : Utilisée ici comme modèle de référence (baseline), elle postule une relation strictement linéaire entre les variables explicatives et la valorisation. Bien qu'elle soit hautement interprétable, son erreur en validation croisée (RMSE de 1786 M$) indique que la dynamique des valorisations est trop complexe et non-linéaire pour être capturée par ce seul algorithme, ce dont on pouvait se douter.

L'Arbre de Décision : Ce modèle présente un cas de surapprentissage (overfitting). Avec une erreur d'entraînement nulle (RMSE de 0.00) et un score R² parfait sur le train set, l'arbre a simplement mémorisé par cœur les données d'entraînement en créant une règle spécifique pour chaque point. L'explosion de son erreur en validation croisée (1951 M$) démontre son incapacité totale à généraliser sur de nouvelles données.

In [ ]:
# Fit a linear regression model to the training set
lin_model = LinearRegression()
lin_model.fit(x_train, y_train)

# Return RMSE for a model and a training set
def compute_error(model, x, y_true):
    # Compute model predictions (median house prices) for training set
    y_pred = model.predict(x)

    # Compute the error between actual and expected median house prices
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse


lin_rmse = compute_error(lin_model, x_train, y_train)
print(f"Training error for linear regression: {lin_rmse:.02f}")

Training error for linear regression: 164.71


In [ ]:
# Fit a decision tree model to the training set
dt_model = DecisionTreeRegressor()
dt_model.fit(x_train, y_train)

dt_rmse = compute_error(dt_model, x_train, y_train)
print(f"Training error for decision tree: {dt_rmse:.02f}")

Training error for decision tree: 0.00


In [ ]:
# Return the mean of cross validation scores for a model and a training set
def compute_crossval_mean_score(model, x, y_true):
    scores = cross_val_score(model, x, y_true, scoring="neg_mean_squared_error", cv=10)
    rmse_scores = np.sqrt(-scores)
    return rmse_scores.mean()


lin_crossval_mean = compute_crossval_mean_score(lin_model, x_train, y_train)
print(f"Mean CV error for linear regression: {lin_crossval_mean:.02f}")

dt_crossval_mean = compute_crossval_mean_score(dt_model, x_train, y_train)
print(f"Mean CV error for decision tree: {dt_crossval_mean:.02f}")

Mean CV error for linear regression: 1786.06
Mean CV error for decision tree: 1951.97


Face aux limites de l'arbre de décision unique, nous avons tenté l'implémentation d'un algorithme de Random Forest (Forêt Aléatoire).

Les résultats de la validation croisée (10 folds) sont meilleurs : l'erreur moyenne chute à 1540 M$. Contrairement à l'arbre unique, le Random Forest réussit à lisser le bruit statistique inhérent aux levées de fonds tout en capturant les relations non-linéaires complexes entre le temps écoulé, le type de série et la croissance de la valorisation.

In [ ]:
# Fit a random forest model to the training set
rf_model = RandomForestRegressor(n_estimators=20)
rf_model.fit(x_train, y_train)

rf_rmse = compute_error(rf_model, x_train, y_train)
print(f"Training error for random forest: {rf_rmse:.02f}")

rf_crossval_mean = compute_crossval_mean_score(rf_model, x_train, y_train)
print(f"Mean CV error for random forest: {rf_crossval_mean:.02f}")

Training error for random forest: 835.94
Mean CV error for random forest: 1540.43


Test du modèle


In [ ]:
# 1. Préparation des données de test
df_x_test = df_test.drop("Post_Val_M", axis=1)
y_test = df_test["Post_Val_M"].to_numpy()

# 2. Nettoyage identique à l'entraînement (on enlève les IDs et dates)
df_x_test_clean = df_x_test.drop(columns=colonnes_a_exclure)

# 3. Application du pipeline pré-entraîné
x_test = full_pipeline.transform(df_x_test_clean)

print(f"Dimensions de x_test : {x_test.shape}")

# 4. Évaluation sur le meilleur modèle (le Random Forest ici)
test_rmse = compute_error(rf_model, x_test, y_test)
print(f"Erreur RMSE de test pour le Random Forest : {test_rmse:.02f} M$")

Dimensions de x_test : (2001, 4628)
Erreur RMSE de test pour le Random Forest : 1884.62 M$


In [ ]:
from sklearn.metrics import r2_score

# 1. Calcul des prédictions sur le jeu d'entraînement
y_train_pred_lin = lin_model.predict(x_train)
y_train_pred_dt = dt_model.predict(x_train)
y_train_pred_rf = rf_model.predict(x_train)

# 2. Calcul des prédictions sur le jeu de test
y_test_pred_lin = lin_model.predict(x_test)
y_test_pred_rf = rf_model.predict(x_test) # On évalue le Random Forest final

# 3. Affichage des résultats
print("=== SCORES R² (ENTRAÎNEMENT) ===")
print(f"Régression Linéaire : {r2_score(y_train, y_train_pred_lin):.4f}")
print(f"Arbre de Décision   : {r2_score(y_train, y_train_pred_dt):.4f}")
print(f"Random Forest       : {r2_score(y_train, y_train_pred_rf):.4f}")

print("\n=== SCORES R² (TEST) ===")
print(f"Régression Linéaire : {r2_score(y_test, y_test_pred_lin):.4f}")
print(f"Random Forest       : {r2_score(y_test, y_test_pred_rf):.4f}")

=== SCORES R² (ENTRAÎNEMENT) ===
Régression Linéaire : 0.9996
Arbre de Décision   : 1.0000
Random Forest       : 0.9892

=== SCORES R² (TEST) ===
Régression Linéaire : 0.9408
Random Forest       : 0.9298


Bien que performant, le Random Forest initial présentait encore des signes de surapprentissage (R² d'entraînement à 0.98 contre 0.92 en test). L'utilisation d'un GridSearchCV a permis de mener une recherche exhaustive pour brider volontairement le modèle (régularisation).

En limitant la profondeur maximale des arbres (max_depth) et en imposant un seuil minimum d'échantillons pour diviser un nœud (min_samples_split), nous avons contraint le modèle à extraire des règles générales plutôt qu'à mémoriser des cas isolés. Les métriques finales illustrent parfaitement le compromis biais-variance : nous acceptons une légère dégradation de la performance sur le jeu d'entraînement pour garantir une prédiction plus robuste et stable sur le jeu de test (RMSE de 1884.62 M$), assurant ainsi la fiabilité du modèle en conditions réelles.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. Définition des hyperparamètres à tester
# C'est ici que l'on "bride" le modèle pour l'empêcher de faire du par cœur
param_grid = {
    'n_estimators': [50, 100, 150],       # Nombre d'arbres dans la forêt
    'max_depth': [5, 10, 15, 20],         # Profondeur maximale (Bloque le surapprentissage)
    'min_samples_split': [2, 5, 10]       # Nombre minimum de startups pour créer une règle
}

# 2. Configuration de la recherche
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,                                 # Validation croisée (découpe les données en 5 pour vérifier)
    scoring='neg_mean_squared_error',
    n_jobs=-1,                            # Utilise tous les cœurs de votre ordinateur pour accélérer
    verbose=1                             # Affiche la barre de progression
)

# 3. Lancement de l'entraînement massif (cela peut prendre quelques minutes)
print("Démarrage de la recherche des paramètres optimaux...")
grid_search.fit(x_train, y_train)

# 4. Extraction du gagnant
best_rf_model = grid_search.best_estimator_
print(f"\nMeilleurs paramètres trouvés : {grid_search.best_params_}")

# 5. Évaluation finale stricte sur le jeu de Test
y_test_pred_best = best_rf_model.predict(x_test)
y_train_pred_best = best_rf_model.predict(x_train)

print("\n=== PERFORMANCES DU MODÈLE OPTIMISÉ ===")
print(f"R² (Entraînement) : {r2_score(y_train, y_train_pred_best):.4f}")
print(f"R² (Test)         : {r2_score(y_test, y_test_pred_best):.4f}")
print(f"RMSE (Test)       : {np.sqrt(mean_squared_error(y_test, y_test_pred_best)):.2f} M$")

Démarrage de la recherche des paramètres optimaux...
Fitting 5 folds for each of 36 candidates, totalling 180 fits


In [1]:
import numpy as np

erreurs = y_test - y_test_pred_best
variance_erreurs = np.var(erreurs)

print("VARIANCE DES ERREURS")
print(f"Variance des résidus sur le test set : {variance_erreurs:.2f}")
print(f"(Rappel - Écart type des erreurs / RMSE : {np.sqrt(variance_erreurs):.2f} M$)\n")

predictions_par_arbre = np.array([arbre.predict(x_test) for arbre in best_rf_model.estimators_])

variance_interne = np.var(predictions_par_arbre, axis=0)
incertitude_ecart_type = np.sqrt(variance_interne)
incertitude_moyenne = np.mean(incertitude_ecart_type)
print(f"\nIncertitude moyenne du modèle sur l'ensemble du test : +/- {incertitude_moyenne:.2f} M$")


for i in range(20) :
  print("EXEMPLE D'INCERTITUDE DU MODÈLE")
  print(f"Vrai montant levé par cette startup : {y_test[i]:.2f} M$")
  print(f"Prédiction du Random Forest (Moyenne) : {y_test_pred_best[i]:.2f} M$")
  print(f"Variance entre les arbres : {variance_interne[i]:.2f}")
  print(f"Incertitude du modèle (+/-) : {incertitude_ecart_type[i]:.2f} M$")

NameError: name 'y_test' is not defined

Dans un contexte d'ingénierie financière, il est intéressant de quantifier le risque et l'incertitude associés à cette prédiction.

L'architecture du Random Forest permet d'extraire cette information : en observant la dispersion (variance) des prédictions générées par chaque arbre individuel de la forêt pour une même startup, nous obtenons un indicateur de confiance empirique. Une forte variance interne indique que le modèle "hésite". Cela signale généralement que la startup évaluée présente un profil atypique (montant levé inhabituel par rapport au délai, etc.) s'éloignant des standards du marché appris par le modèle. Cette métrique d'incertitude (ici mesurée en écart-type de +/- M$) est un outil d'aide à la décision précieux.

Malgré l'application rigoureuse d'un pipeline de préparation des données et d'une recherche d'hyperparamètres, nos premiers modèles (Régression Linéaire, Arbre de Décision, Random Forest) se heurtent à des limites structurelles inhérentes à notre problématique.

Le problème fondamental réside dans la nature séquentielle et temporelle de nos données. Nous cherchons à prédire la valorisation d'une start-up au tour n+1 en fonction de son historique (tours 1 à n). Or, ces trois algorithmes classiques considèrent chaque ligne du jeu de données comme une observation indépendante et isolée (hypothèse I.I.D), ignorant la dimension chronologique de la croissance :

La Régression Linéaire : Ce modèle est beaucoup trop rigide. Il postule que l'ajout d'un mois entre deux levées de fonds ou d'un million de dollars levés aura un impact additif constant sur la valorisation, quel que soit le stade de maturité de l'entreprise. Il est mathématiquement incapable de modéliser l'effet multiplicateur et exponentiel caractéristique de l'hyper-croissance d'une start-up.

L'Arbre de Décision : Comme démontré par nos scores, ce modèle fragmente l'espace des données avec des règles binaires rigides, menant à un surapprentissage total. Il mémorise des trajectoires spécifiques sans en comprendre la dynamique sous-jacente.

Le Random Forest : Bien qu'il corrige le surapprentissage de l'arbre unique en réduisant la variance, il présente deux failles critiques pour la prédiction financière :

L'incapacité d'extrapolation : Un Random Forest effectue ses prédictions en calculant la moyenne des valeurs présentes dans ses feuilles terminales. Par conséquent, il est structurellement incapable de prédire une valorisation strictement supérieure au maximum observé dans son jeu d'entraînement. Face à une start-up entrant dans une phase d'hyper-croissance inédite, le modèle plafonnera systématiquement sa prédiction.

L'absence de mémoire temporelle : Il traite les données de manière statique. Il ne comprend pas nativement qu'une levée de fonds en Series B est la suite chronologique et logique de la Series A de cette même entreprise, perdant ainsi toute notion de "momentum" (dynamique de croissance).

Face à ces constats, il est nécessaire de pivoter vers des architectures capables de modéliser des séquences ou d'optimiser l'erreur de manière asymétrique.